<a href="https://colab.research.google.com/github/nisarg2806/heloo/blob/main/AI_BASED_HIRING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI BASED HIRING PRIDICTION

# TASK 1 Load and Understand the Dataset

In [123]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('/content/AI-Based Hiring Prediction System.csv')
print("Dataset loaded successfully")

Dataset loaded successfully


In [124]:
df.head()
df.tail()
df.sample(5)

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
864,865,Joseph Gallagher,"Networking, Cybersecurity, Ethical Hacking",8,B.Tech,NaN,Cybersecurity Analyst,Hire,94140,4,100
254,255,Sara Collins,"React, C++, SQL, Java",7,MBA,Deep Learning Specialization,Software Engineer,Hire,90437,5,100
758,759,Eric Jimenez,"NLP, TensorFlow, Pytorch, Python",6,B.Tech,Google ML,AI Researcher,Hire,75755,0,90
20,21,Nichole Welch,"SQL, Java, C++, React",5,B.Sc,NaN,Software Engineer,Hire,57546,4,90
215,216,Gloria Miller,"Linux, Ethical Hacking, Cybersecurity, Networking",8,M.Tech,AWS Certified,Cybersecurity Analyst,Hire,45438,3,100


**Explanation**





The dataset contains structured and unstructured data.

Structured: Experience, Salary, Projects Count

Unstructured: Skills, Certifications, Job Role

The target variable is: Recruiter Decision

We will convert:
Hire → 1
Reject → 0

# TASK 2: Basic Data Inspection

In [125]:
df.shape
df.columns
df.dtypes
df["Recruiter Decision"].value_counts()
df.describe()

,Resume_ID,Experience (Years),Salary Expectation ($),Projects Count,AI Score (0-100)
count,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,500.500000,4.896000,79994.486000,5.13300,83.950000
std,288.819436,3.112695,23048.472549,3.23137,20.983036
min,1.000000,0.000000,40085.000000,0.00000,15.000000
25%,250.750000,2.000000,60415.750000,2.00000,70.000000
50%,500.500000,5.000000,79834.500000,5.00000,100.000000
75%,750.250000,8.000000,99583.250000,8.00000,100.000000
max,1000.000000,10.000000,119901.000000,10.00000,100.000000


**Why Data Inspection is Important?**

Helps understand dataset structure

Detects missing values

Checks class imbalance

Identifies incorrect data types

Prevents errors during model training

# TASK 3: Data Cleaning & Preprocessing

**Drop Unnecessary Columns**

In [126]:
df.drop(["Resume_ID", "Name"], axis=1, inplace=True)

**Check Missing Values**

In [127]:
df.isnull().sum()

,0
Skills,0
Experience (Years),0
Education,0
Certifications,274
Job Role,0
Recruiter Decision,0
Salary Expectation ($),0
Projects Count,0
AI Score (0-100),0


**If missing:**

Text → fill with ""

Numeric → fill with mean/median

**Ensure Numeric Format**

In [128]:
df["Experience (Years)"] = pd.to_numeric(df["Experience (Years)"])
df["Salary Expectation ($)"] = pd.to_numeric(df["Salary Expectation ($)"])
df["Projects Count"] = pd.to_numeric(df["Projects Count"])

# TASK 4: Text Feature Engineering

**Combine Text Columns**

In [129]:
df["combined_text"] = (
    df["Skills"] + " " +
    df["Certifications"] + " " +
    df["Job Role"]
)

**Clean Text**

In [130]:
import re
import pandas as pd


def clean_text(text):
    if pd.isnull(text):
       text = ""
    else:
        text = str(text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["combined_text"] = df["combined_text"].apply(clean_text)

**Why Text Cleaning?**

Removes noise

Reduces duplicate tokens

Improves vectorization quality

Makes model more accurate

# TASK 5: Convert Text to Numerical Features

In [131]:
tfidf = TfidfVectorizer(max_features=500)
text_features = tfidf.fit_transform(df["combined_text"])

**Why ML cannot use text directly?**

ML models work on numbers, not strings.

What is TF-IDF?

TF-IDF = Term Frequency × Inverse Document Frequency
It measures:

How important a word is in one document

Compared to all other documents

# TASK 6: Encode Categorical Variable

**Encode Education:**

In [132]:
le = LabelEncoder()
df["Education"] = le.fit_transform(df["Education"])

# TASK 7: Feature & Target Separation

In [133]:
X_numeric = df[["Experience (Years)", "Salary Expectation ($)", "Projects Count", "Education"]]
y = df["Recruiter Decision"]

**Combine text + numeric:**

In [134]:
from scipy.sparse import hstack
X = hstack((text_features, X_numeric))

**Why not include target in X?**

Because the model would "cheat" and already know the answer.

# TASK 8: Train-Test Split

In [135]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Why Train-Test Split?**

Train → learn patterns

Test → evaluate performance

Prevents overfitting

What is Overfitting?

When model performs well on training data
But poorly on new/unseen data.

# TASK 9: Feature Scaling

In [136]:
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Why Scaling?**

Makes features comparable

Required for distance-based models

Models that require scaling:

Logistic Regression

SVM

KNN

Random Forest does NOT require scaling.

# TASK 10: Model Training

**Logistic Regression**

In [137]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)

LogisticRegression()

**How it works:**

Uses sigmoid function to predict probabilit

**Random Forest**

In [138]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

**How it works:**

Builds multiple decision trees and averages results.

**Support Vector Machine**

In [139]:
from sklearn.svm import SVC
svm = SVC(probability=True)
svm.fit(X_train_scaled, y_train)

SVC(probability=True)

**How it works:**

Finds optimal boundary (hyperplane) between classes.

**K-Nearest Neighbors**

In [140]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)

KNeighborsClassifier()

**How it works:**

Predicts based on majority class of nearest neighbors.

# TASK 11: Model Evaluation

In [141]:
models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "SVM": svm,
    "KNN": knn
}

for name, model in models.items():
    if name == "Random Forest":
        y_pred = model.predict(X_test)
    else:
        y_pred = model.predict(X_test_scaled)

    print(name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("----------------------------------")

Logistic Regression
Accuracy: 0.98
              precision    recall  f1-score   support

        Hire       0.98      0.99      0.99       154
      Reject       0.98      0.93      0.96        46

    accuracy                           0.98       200
   macro avg       0.98      0.96      0.97       200
weighted avg       0.98      0.98      0.98       200

----------------------------------
Random Forest
Accuracy: 0.955
              precision    recall  f1-score   support

        Hire       0.95      0.99      0.97       154
      Reject       0.97      0.83      0.89        46

    accuracy                           0.95       200
   macro avg       0.96      0.91      0.93       200
weighted avg       0.96      0.95      0.95       200

----------------------------------
SVM
Accuracy: 0.895
              precision    recall  f1-score   support

        Hire       0.88      0.99      0.94       154
      Reject       0.96      0.57      0.71        46

    accuracy               

**Create Comparison Table:**

In [142]:
results = []

for name, model in models.items():
    if name == "Random Forest":
        y_pred = model.predict(X_test)
    else:
        y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    results.append([name, acc])

pd.DataFrame(results, columns=["Model Name", "Accuracy"])

,Model Name,Accuracy
0,Logistic Regression,0.980
1,Random Forest,0.955
2,SVM,0.895
3,KNN,0.850


**Which Model Performs Best?**

Usually:

Random Forest performs best for mixed text + numeric features

Because it handles nonlinear relationships well

# TASK 12: Pipeline & GridSearch (Advanced)

In [143]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=500)),
    ("clf", LogisticRegression())
])

params = {
    "clf__C": [0.1, 1, 10]
}

grid = GridSearchCV(pipeline, params, cv=5)
grid.fit(df["combined_text"], y)

print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Params: {'clf__C': 0.1}
Best Score: 0.812


**Why Pipelines?**

Automates preprocessing + model

Prevents data leakage

Used in production systems

# TASK 13: Hiring Prediction Function

In [144]:
def predict_candidate(skills, experience, education, certifications, projects, salary):

    text = clean_text(skills + " " + certifications)
    text_vector = tfidf.transform([text])

    edu = le.transform([education])[0]

    numeric = np.array([[experience, salary, projects, edu]])

    from scipy.sparse import hstack
    final_input = hstack((text_vector, numeric))

    final_input_scaled = scaler.transform(final_input)

    prediction = lr.predict(final_input_scaled)[0]
    probability = lr.predict_proba(final_input_scaled)[0][1]

    if prediction == 1:
        result = "Hire"
    else:
        result = "Reject"

    return result, probability

# TASK 14: Final Conclusion

**Dataset Understanding**

Mixed structured and unstructured resume data

Target: Recruiter Decision

**Key Preprocessing Steps**

Dropped irrelevant columns

Converted target to numeric

Cleaned text

Used TF-IDF

Encoded categorical features

Scaled numeric features

**Best Model**

(Random Forest or SVM depending on accuracy)

** What You Learned**

Handling text + numeric features

Feature engineering

Model comparison

Overfitting & scaling

Building prediction function

**Real-World Connection**

This project simulates:

AI Resume Screening

HR Automation

Talent Acquisition Analytics

Smart Recruitment Systems

Companies use similar ML pipelines in real hiring systems.